# Three Peak Demo 1

This notebook targets the `examples/three-peak-demo1` workspace. It regenerates the sequence JSON files from the current Q1ASM sources, configures one QCM plus two QRMs, routes LINQ feedback, and starts the eight sequencers used by the demo. The peaks use independent bounded drift personalities so Ch1, Ch2, and Ch3 move separately without wrap-around jumps.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import scipy.signal
from qcodes.instrument import find_or_create_instrument
from qblox_instruments import Cluster, ClusterType

ROOT = Path.cwd()
assert (ROOT / "q1timeline.yml").exists(), "Run this notebook from examples/three-peak-demo1"

def load_params() -> dict[str, int]:
    return json.loads((ROOT / "params.json").read_text(encoding="utf-8"))

def render_q1asm(name: str, params: dict[str, int]) -> str:
    return (ROOT / name).read_text(encoding="utf-8").format_map(params)

params = load_params()
params

## Connect to the cluster

Set `cluster_ip` for hardware. Set it to `None` to use the dummy configuration.

In [ ]:
cluster_ip = "10.10.200.52"
cluster_name = "QAS-DC-1"

qcm_module_index = 0
qrm0_module_index = 0
qrm1_module_index = 1

dummy_cfg = (
    {
        2: ClusterType.CLUSTER_QCM,
        4: ClusterType.CLUSTER_QRM,
        6: ClusterType.CLUSTER_QRM,
    }
    if cluster_ip is None
    else None
)

cluster: Cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name=cluster_name,
    identifier=cluster_ip,
    dummy_cfg=dummy_cfg,
    debug=1,
)

cluster.reset()

qcm_modules = cluster.get_connected_modules(lambda mod: mod.is_qcm_type and not mod.is_rf_type)
qrm_modules = cluster.get_connected_modules(lambda mod: mod.is_qrm_type and not mod.is_rf_type)

qcm_module = qcm_modules[qcm_module_index]
qrm0_module = qrm_modules[qrm0_module_index]
qrm1_module = qrm_modules[qrm1_module_index]

print(cluster.get_system_status())
print("QCM:", qcm_module)
print("QRM0:", qrm0_module)
print("QRM1:", qrm1_module)

## Generate sequence JSON

This cell mirrors the checked-in three-peak example: QCM sequencer 0 emits Ch0 sum plus Ch1, QCM sequencer 1 emits Ch2 plus Ch3, QRM0 tracks Ch1 and Ch2, and QRM1 tracks Ch3. The Q1ASM drift code uses independent bounded drift personalities for the three peak centers.

In [ ]:
T_TOTAL = params["T_TOTAL"]
GAUSS_DUR = params["GAUSS_DUR"]
CURSOR_DUR = params["CURSOR_DUR"]
ACQ_DUR = params["ACQ_DUR"]

cluster.clear_router()
cluster.set_cmm_route(params["CH1_DELAY_ID"], [qrm0_module.sequencer0])
cluster.set_cmm_route(params["CH1_GAIN_ID"], [qrm0_module.sequencer0])
cluster.set_cmm_route(params["CH2_DELAY_ID"], [qrm0_module.sequencer2])
cluster.set_cmm_route(params["CH2_GAIN_ID"], [qrm0_module.sequencer2])
cluster.set_cmm_route(params["CH3_DELAY_ID"], [qrm1_module.sequencer0])
cluster.set_cmm_route(params["CH3_GAIN_ID"], [qrm1_module.sequencer0])

gaussian_pulse = scipy.signal.windows.gaussian(GAUSS_DUR, std=0.12 * GAUSS_DUR).tolist()
scope_trigger_pulse = [1.0] * params["SCOPE_TRIG_DUR"]
square_pulse = [1.0] * CURSOR_DUR

def tracker_acquisitions(prefix: str) -> dict[str, dict[str, int]]:
    return {
        f"{prefix.lower()}_edge": {"num_bins": 2, "index": params[f"{prefix}_EDGE_ACQ"]},
        f"{prefix.lower()}_position": {"num_bins": 256, "index": params[f"{prefix}_POS_ACQ"]},
    }

sequences = {
    "qcm_sum_ch1_sequence.json": {
        "waveforms": {
            "gaussian": {"data": gaussian_pulse, "index": 0},
            "trigger": {"data": scope_trigger_pulse, "index": 1},
        },
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("qcm_sum_ch1.q1asm", params),
    },
    "qcm_ch2_ch3_sequence.json": {
        "waveforms": {"gaussian": {"data": gaussian_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("qcm_ch2_ch3.q1asm", params),
    },
    "qrm0_ch1_probe_sequence.json": {
        "waveforms": {"square": {"data": square_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("qrm0_ch1_probe.q1asm", params),
    },
    "qrm0_ch1_tracker_sequence.json": {
        "waveforms": {},
        "weights": {},
        "acquisitions": tracker_acquisitions("CH1"),
        "program": render_q1asm("qrm0_ch1_tracker.q1asm", params),
    },
    "qrm0_ch2_probe_sequence.json": {
        "waveforms": {"square": {"data": square_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("qrm0_ch2_probe.q1asm", params),
    },
    "qrm0_ch2_tracker_sequence.json": {
        "waveforms": {},
        "weights": {},
        "acquisitions": tracker_acquisitions("CH2"),
        "program": render_q1asm("qrm0_ch2_tracker.q1asm", params),
    },
    "qrm1_ch3_probe_sequence.json": {
        "waveforms": {"square": {"data": square_pulse, "index": 0}},
        "weights": {},
        "acquisitions": {},
        "program": render_q1asm("qrm1_ch3_probe.q1asm", params),
    },
    "qrm1_ch3_tracker_sequence.json": {
        "waveforms": {},
        "weights": {},
        "acquisitions": tracker_acquisitions("CH3"),
        "program": render_q1asm("qrm1_ch3_tracker.q1asm", params),
    },
}

for file_name, sequence in sequences.items():
    (ROOT / file_name).write_text(json.dumps(sequence, indent=4), encoding="utf-8")

print("Wrote", ", ".join(sequences))

## Optional timeline check

This validates the notebook-generated files against the local `q1timeline` analyzer before loading hardware.

In [ ]:
out_dir = ROOT / ".q1timeline"
out_dir.mkdir(exist_ok=True)

env = os.environ.copy()
repo_root = ROOT.parents[1]
env["PYTHONPATH"] = str(repo_root / "src")

subprocess.run(
    [
        sys.executable,
        "-m",
        "q1timeline",
        "analyze",
        "--project",
        "q1timeline.yml",
        "--out",
        str(out_dir / "timeline_ir.json"),
        "--diagnostics",
        str(out_dir / "diagnostics.json"),
    ],
    cwd=ROOT,
    env=env,
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "q1timeline",
        "render",
        "--ir",
        str(out_dir / "timeline_ir.json"),
        "--out",
        str(out_dir / "timeline.html"),
    ],
    cwd=ROOT,
    env=env,
    check=True,
)

## Configure and start sequencers

The loop speed is controlled externally through trigger address 1. The Q1ASM wait budget includes the trigger wait, so keep the trigger period at or above `T_TOTAL`.

In [ ]:
cluster.ext_trigger_input_trigger_en(True)
cluster.ext_trigger_input_trigger_address(1)

qcm_module.stop_sequencer()
qrm0_module.stop_sequencer()
qrm1_module.stop_sequencer()

qcm_module.disconnect_outputs()
qrm0_module.disconnect_outputs()
qrm0_module.disconnect_inputs()
qrm1_module.disconnect_outputs()
qrm1_module.disconnect_inputs()

qcm_module.sequencer0.connect_sequencer("out0_1")
qcm_module.sequencer1.connect_sequencer("out2_3")
qrm0_module.sequencer0.connect_sequencer("io0_1")
qrm0_module.sequencer1.connect_sequencer("io0_1")
qrm0_module.sequencer2.connect_sequencer("io0_1")
qrm0_module.sequencer3.connect_sequencer("io0_1")
qrm1_module.sequencer0.connect_sequencer("io0_1")
qrm1_module.sequencer1.connect_sequencer("io0_1")

qrm0_module.sequencer1.demod_en_acq(True)
qrm0_module.sequencer1.integration_length_acq(ACQ_DUR)
qrm0_module.sequencer3.demod_en_acq(True)
qrm0_module.sequencer3.integration_length_acq(ACQ_DUR)
qrm1_module.sequencer1.demod_en_acq(True)
qrm1_module.sequencer1.integration_length_acq(ACQ_DUR)
qrm0_module.sequencer0.mod_en_awg(False)
qrm0_module.sequencer2.mod_en_awg(False)
qrm1_module.sequencer0.mod_en_awg(False)

qcm_module.sequencer0.sequence("qcm_sum_ch1_sequence.json")
qcm_module.sequencer1.sequence("qcm_ch2_ch3_sequence.json")
qrm0_module.sequencer0.sequence("qrm0_ch1_probe_sequence.json")
qrm0_module.sequencer1.sequence("qrm0_ch1_tracker_sequence.json")
qrm0_module.sequencer2.sequence("qrm0_ch2_probe_sequence.json")
qrm0_module.sequencer3.sequence("qrm0_ch2_tracker_sequence.json")
qrm1_module.sequencer0.sequence("qrm1_ch3_probe_sequence.json")
qrm1_module.sequencer1.sequence("qrm1_ch3_tracker_sequence.json")

for seq in (
    qcm_module.sequencer0,
    qcm_module.sequencer1,
    qrm0_module.sequencer0,
    qrm0_module.sequencer1,
    qrm0_module.sequencer2,
    qrm0_module.sequencer3,
    qrm1_module.sequencer0,
    qrm1_module.sequencer1,
):
    seq.sync_en(True)

qcm_module.arm_sequencer(0)
qcm_module.arm_sequencer(1)
qrm0_module.arm_sequencer(0)
qrm0_module.arm_sequencer(1)
qrm0_module.arm_sequencer(2)
qrm0_module.arm_sequencer(3)
qrm1_module.arm_sequencer(0)
qrm1_module.arm_sequencer(1)

qcm_module.start_sequencer()
qrm0_module.start_sequencer()
qrm1_module.start_sequencer()

print(qcm_module.get_sequencer_status(0))
print(qcm_module.get_sequencer_status(1))
print(qrm0_module.get_sequencer_status(0))
print(qrm0_module.get_sequencer_status(1))
print(qrm0_module.get_sequencer_status(2))
print(qrm0_module.get_sequencer_status(3))
print(qrm1_module.get_sequencer_status(0))
print(qrm1_module.get_sequencer_status(1))

## Stop and reset

In [ ]:
qcm_module.stop_sequencer()
qrm0_module.stop_sequencer()
qrm1_module.stop_sequencer()

print(qcm_module.get_sequencer_status(0))
print(qcm_module.get_sequencer_status(1))
print(qrm0_module.get_sequencer_status(0))
print(qrm0_module.get_sequencer_status(1))
print(qrm0_module.get_sequencer_status(2))
print(qrm0_module.get_sequencer_status(3))
print(qrm1_module.get_sequencer_status(0))
print(qrm1_module.get_sequencer_status(1))

cluster.reset()
print(cluster.get_system_status())